In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=2db7276036c9d7231845724dce38dc32b3360b1105206914ee32f2934c5c567b
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
# ===========================
# Colab-safe ExCIR benchmark
# ===========================

# Install first in Colab:
# !pip install numpy pandas scipy scikit-learn tqdm
# !pip install torch torchvision
# !pip install shap lime scikit-image

import os
import json
import time
import random
import warnings
import collections
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cross_decomposition import CCA
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

import torchvision
from torchvision import datasets, models, transforms

# ---------------------------
# Optional packages
# ---------------------------
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

try:
    from lime import lime_image
    from lime.lime_text import LimeTextExplainer
    HAS_LIME = True
except Exception:
    HAS_LIME = False

# ---------------------------
# Config: change here only
# ---------------------------
CONFIG = {
    "track": "vision",              # "vision" or "text"
    "data_dir": "./data",
    "out_dir": "./results_revision",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    # generic
    "run_lw": True,
    "run_shap": False,             # start False first
    "run_lime": False,             # start False first
    "topk": 10,
    "n_boot": 30,

    # vision
    "epochs": 15,
    "epochs_lw": 5,
    "lr": 3e-4,
    "batch_size": 128,
    "train_subset": None,          # increase later
    "val_subset": 5000,
    "test_subset": 5000,
    "patch_size": 4,
    "image_noise_sigma": 0.10,
    "deletion_steps": 10,

    # text
    "text_max_docs": 6000,
    "max_text_features": 10000,
    "text_drop_prob": 0.15,
    "text_lime_features": 20,

    # explainers
    "shap_samples": 64,
    "lime_samples": 16,
    "lime_num_samples": 300,

    # LW thresholds
    "proj_threshold": 0.20,
    "mmd_threshold": 0.10,
    "kl_threshold": 0.10,
    "risk_gap_threshold": 0.03,
    "lw_spearman_threshold": 0.95,
    "lw_topk_threshold": 1.00,
}

# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

# ---------------------------
# ExCIR core
# ---------------------------
def cir_score_1d(x, y, eps=1e-12):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    if len(x) != len(y):
        raise ValueError("x and y must have same length")
    x_hat = x.mean()
    y_hat = y.mean()
    m = 0.5 * (x_hat + y_hat)
    num = len(x) * ((x_hat - m) ** 2 + (y_hat - m) ** 2)
    den = np.sum((x - m) ** 2) + np.sum((y - m) ** 2) + eps
    return float(num / den)

def excir_scores(X, Y, weights=None):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if Y.ndim == 1:
        Y = Y[:, None]
    n, d = X.shape
    _, c = Y.shape
    if weights is None:
        weights = np.ones(c) / c
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / (weights.sum() + 1e-12)

    scores = np.zeros(d, dtype=np.float64)
    for j in range(d):
        scores[j] = sum(weights[l] * cir_score_1d(X[:, j], Y[:, l]) for l in range(c))
    return scores

def cc_cir_scores(X, logits, class_index):
    vc = logits[:, class_index]
    return np.array([cir_score_1d(X[:, j], vc) for j in range(X.shape[1])])

def _safe_standardize(X, eps=1e-8):
    X = np.asarray(X, dtype=np.float64)
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd = np.where(sd < eps, 1.0, sd)
    return (X - mu) / sd

def blockcir_scores(X, Y, groups):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if Y.ndim == 1:
        Y = Y[:, None]

    out = {}
    for name, idxs in groups.items():
        idxs = list(idxs)
        Xg = _safe_standardize(X[:, idxs])

        if Y.shape[1] == 1:
            y = Y[:, 0]
            cov = Xg.T @ y
            if np.allclose(cov, 0):
                z = Xg.mean(axis=1)
            else:
                w = cov / (np.linalg.norm(cov) + 1e-12)
                z = Xg @ w
            out[name] = cir_score_1d(z, y)
        else:
            try:
                cca = CCA(n_components=1, max_iter=1000)
                Zx, Zy = cca.fit_transform(Xg, Y)
                out[name] = cir_score_1d(Zx[:, 0], Zy[:, 0])
            except Exception:
                out[name] = float(excir_scores(Xg, Y).mean())
    return out

def rank_indices(scores):
    return np.argsort(-np.asarray(scores))

def topk_overlap(a, b, k=10):
    sa, sb = set(a[:k]), set(b[:k])
    return len(sa & sb) / max(len(sa | sb), 1)

def rank_df(scores, names):
    df = pd.DataFrame({"feature": list(names), "score": np.asarray(scores)})
    return df.sort_values("score", ascending=False).reset_index(drop=True)

def timed(fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    t1 = time.perf_counter()
    return out, (t1 - t0)

# ---------------------------
# Stability
# ---------------------------
def bootstrap_ranking_stability(X, Y, score_fn, n_boot=30, topk=10, seed=42):
    rng = np.random.default_rng(seed)
    base_scores = score_fn(X, Y)
    base_rank = rank_indices(base_scores)

    spearmans, kendalls, overlaps = [], [], []
    n = X.shape[0]

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        bs_scores = score_fn(X[idx], Y[idx])
        bs_rank = rank_indices(bs_scores)
        spearmans.append(np.nan_to_num(stats.spearmanr(base_scores, bs_scores).statistic))
        kendalls.append(np.nan_to_num(stats.kendalltau(base_rank, bs_rank).statistic))
        overlaps.append(topk_overlap(base_rank, bs_rank, k=topk))

    return {
        "spearman_mean": float(np.mean(spearmans)),
        "kendall_mean": float(np.mean(kendalls)),
        "topk_overlap_mean": float(np.mean(overlaps)),
        "topk_overlap_std": float(np.std(overlaps)),
    }

# ---------------------------
# LW environment checks
# ---------------------------
class LWCriteria:
    def __init__(
        self,
        proj_threshold=0.20,
        mmd_threshold=0.10,
        kl_threshold=0.10,
        risk_gap_threshold=0.03,
        spearman_threshold=0.95,
        topk_threshold=1.00,
    ):
        self.proj_threshold = proj_threshold
        self.mmd_threshold = mmd_threshold
        self.kl_threshold = kl_threshold
        self.risk_gap_threshold = risk_gap_threshold
        self.spearman_threshold = spearman_threshold
        self.topk_threshold = topk_threshold

def projection_distance(X_full, X_lw, k=10):
    X_full = _safe_standardize(X_full)
    X_lw = _safe_standardize(X_lw)
    k = min(k, X_full.shape[1], X_lw.shape[1], X_full.shape[0] - 1, X_lw.shape[0] - 1)
    if k <= 0:
        return 0.0
    _, _, vf = np.linalg.svd(X_full, full_matrices=False)
    _, _, vl = np.linalg.svd(X_lw, full_matrices=False)
    sf = vf[:k].T @ vf[:k]
    sl = vl[:k].T @ vl[:k]
    return float(np.linalg.norm(sf - sl, ord="fro") / max(k, 1))

def rbf_mmd(X, Y, gamma=None):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if gamma is None:
        gamma = 1.0 / max(X.shape[1], 1)

    def kernel(A, B):
        d2 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)
        return np.exp(-gamma * d2)

    Kxx = kernel(X, X)
    Kyy = kernel(Y, Y)
    Kxy = kernel(X, Y)
    return float(Kxx.mean() + Kyy.mean() - 2 * Kxy.mean())

def approx_gaussian_kl(X_full, X_lw, eps=1e-6):
    mu0 = X_full.mean(axis=0)
    mu1 = X_lw.mean(axis=0)
    var0 = X_full.var(axis=0) + eps
    var1 = X_lw.var(axis=0) + eps
    kl = 0.5 * np.sum(np.log(var1 / var0) + (var0 + (mu0 - mu1) ** 2) / var1 - 1.0)
    return float(max(kl, 0.0))

def evaluate_lw_gate(X_full, X_lw, full_scores, lw_scores, full_metric, lw_metric, criteria, topk=8):
    proj = projection_distance(X_full, X_lw)
    mmd = rbf_mmd(X_full[: min(512, len(X_full))], X_lw[: min(512, len(X_lw))])
    kl = approx_gaussian_kl(X_full, X_lw)
    rho = np.nan_to_num(stats.spearmanr(full_scores, lw_scores).statistic)
    overlap = topk_overlap(rank_indices(full_scores), rank_indices(lw_scores), k=topk)
    risk_gap = abs(full_metric - lw_metric)

    passed = (
        proj <= criteria.proj_threshold and
        mmd <= criteria.mmd_threshold and
        kl <= criteria.kl_threshold and
        risk_gap <= criteria.risk_gap_threshold and
        rho >= criteria.spearman_threshold and
        overlap >= criteria.topk_threshold
    )

    return {
        "proj": float(proj),
        "mmd": float(mmd),
        "kl": float(kl),
        "spearman": float(rho),
        "topk_overlap": float(overlap),
        "risk_gap": float(risk_gap),
        "passed": bool(passed),
    }

# ---------------------------
# Vision: CIFAR-10 + ResNet18
# ---------------------------
class WrappedSubset(Dataset):
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        x, y = self.base[idx]
        return x, y, idx

def make_resnet18(num_classes=10):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def make_cifar_dataloaders(data_dir, batch_size=128, train_subset=None, val_subset=None, test_subset=None, seed=42):
    tfm = transforms.Compose([transforms.ToTensor()])
    train_all = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=tfm)
    test_all = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=tfm)

    idx = np.arange(len(train_all))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    split = int(0.9 * len(idx))
    tr_idx, va_idx = idx[:split], idx[split:]

    if train_subset is not None:
        tr_idx = tr_idx[:train_subset]
    if val_subset is not None:
        va_idx = va_idx[:val_subset]
    te_idx = np.arange(len(test_all))
    if test_subset is not None:
        te_idx = te_idx[:test_subset]

    train_ds = WrappedSubset(Subset(train_all, tr_idx))
    val_ds = WrappedSubset(Subset(train_all, va_idx))
    test_ds = WrappedSubset(Subset(test_all, te_idx))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader

def train_torch_classifier(model, train_loader, val_loader, device, epochs=2, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_val = -1.0
    history = []

    for epoch in range(epochs):
        model.train()
        losses = []
        for xb, yb, *_ in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        val_acc = eval_torch_accuracy(model, val_loader, device)
        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "val_acc": float(val_acc)
        })
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, {"best_val_acc": float(best_val), "history": history}

@torch.no_grad()
def eval_torch_accuracy(model, loader, device):
    model.eval()
    ys, yh = [], []
    for xb, yb, *_ in loader:
        logits = model(xb.to(device))
        pred = logits.argmax(dim=1).cpu().numpy()
        ys.append(yb.numpy())
        yh.append(pred)
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(yh)
    return float(accuracy_score(y_true, y_pred))

@torch.no_grad()
def collect_logits_and_inputs(model, loader, device):
    model.eval()
    Xs, ys, logits_all = [], [], []
    for xb, yb, *_ in loader:
        logits = model(xb.to(device)).cpu().numpy()
        Xs.append(xb.numpy())
        ys.append(yb.numpy())
        logits_all.append(logits)
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(logits_all)

def images_to_patch_features(X, patch=4, channel_reduce="keep"):
    n, c, h, w = X.shape
    assert h % patch == 0 and w % patch == 0
    feats, names = [], []
    groups = collections.defaultdict(list)
    col = 0

    if channel_reduce == "mean":
        Xr = X.mean(axis=1, keepdims=True)
        c = 1
    else:
        Xr = X

    for ch in range(c):
        for i in range(0, h, patch):
            for j in range(0, w, patch):
                vals = Xr[:, ch, i:i+patch, j:j+patch].mean(axis=(1, 2))
                feats.append(vals)
                names.append(f"ch{ch}_p{i//patch}_{j//patch}")
                groups[f"region_{i//patch}_{j//patch}"].append(col)
                col += 1

    return np.stack(feats, axis=1), names, dict(groups)

def add_gaussian_noise_images(X, sigma=0.1):
    Xn = X + np.random.normal(0, sigma, size=X.shape)
    return np.clip(Xn, 0.0, 1.0)

# ---------------------------
# SHAP / LIME
# ---------------------------
def shap_text_scores_linear(clf, X_val_sparse, sample_size=300):
    if not HAS_SHAP:
        raise ImportError("shap not installed")

    sample_size = min(sample_size, X_val_sparse.shape[0])
    background = X_val_sparse[: min(200, X_val_sparse.shape[0])]
    Xs = X_val_sparse[:sample_size]

    explainer = shap.LinearExplainer(clf, background)
    vals = explainer.shap_values(Xs)

    # Case 1: old SHAP returns list[class] of (n_samples, n_features)
    if isinstance(vals, list):
        arr = np.mean([np.abs(v).mean(axis=0) for v in vals], axis=0)

    # Case 2: newer SHAP may return (n_samples, n_features, n_classes)
    elif vals.ndim == 3:
        arr = np.abs(vals).mean(axis=(0, 2))

    # Case 3: binary/single-output returns (n_samples, n_features)
    else:
        arr = np.abs(vals).mean(axis=0)

    return np.asarray(arr).reshape(-1)

def lime_text_global_scores(clf, vec, texts, class_names, num_samples=50, num_features=20, seed=42):
    if not HAS_LIME:
        raise ImportError("lime not installed")

    explainer = LimeTextExplainer(class_names=list(class_names), random_state=seed)
    vocab = vec.get_feature_names_out()
    vocab_to_idx = {w: i for i, w in enumerate(vocab)}
    scores = np.zeros(len(vocab), dtype=np.float64)

    def predict_fn(raw_texts):
        X = vec.transform(raw_texts)
        return clf.predict_proba(X)

    valid_texts = [t for t in texts if isinstance(t, str) and len(t.split()) >= 2]
    num_samples = min(num_samples, len(valid_texts))

    if num_samples == 0:
        return scores

    for i in tqdm(range(num_samples), desc="LIME-text", leave=False):
        try:
            exp = explainer.explain_instance(
                valid_texts[i],
                predict_fn,
                num_features=num_features
            )
            for word, weight in exp.as_list():
                word = word.lower()
                if word in vocab_to_idx:
                    scores[vocab_to_idx[word]] += abs(weight)
        except Exception:
            continue

    return scores / max(num_samples, 1)

def shap_image_patch_scores(model, background, images, device, patch_size=4, sample_size=32):
    if not HAS_SHAP:
        raise ImportError("shap not installed")

    model.eval()
    bg = background[: min(len(background), 16)].to(device)
    xs = images[: min(len(images), sample_size)].to(device)

    explainer = shap.GradientExplainer(model, bg)
    vals = explainer.shap_values(xs)

    # Old SHAP: list of arrays, one per class
    if isinstance(vals, list):
        arr = np.mean([np.abs(v) for v in vals], axis=0)

    # New SHAP: array shape can be (n, c, h, w, classes)
    else:
        arr = np.asarray(vals)
        if arr.ndim == 5:
            arr = np.abs(arr).mean(axis=-1)  # average over classes
        else:
            arr = np.abs(arr)

    # Now arr should be (n, c, h, w)
    if arr.ndim != 4:
        raise ValueError(f"Unexpected SHAP image shape after processing: {arr.shape}")

    arr = arr.mean(axis=0, keepdims=True)  # (1, c, h, w)

    patch_scores, _, _ = images_to_patch_features(
        arr, patch=patch_size, channel_reduce="keep"
    )

    return patch_scores.reshape(-1)

def lime_image_patch_scores(model, images, device, patch_size=4, num_samples=8, lime_num_samples=200):
    if not HAS_LIME:
        raise ImportError("lime not installed")
    explainer = lime_image.LimeImageExplainer()
    scores = None
    images = images[: min(len(images), num_samples)]

    def predict_fn(imgs):
        arr = imgs.astype(np.float32)
        if arr.max() > 1.0:
            arr = arr / 255.0
        tens = torch.from_numpy(arr).permute(0, 3, 1, 2).to(device)
        with torch.no_grad():
            probs = F.softmax(model(tens), dim=1).cpu().numpy()
        return probs

    for img in tqdm(images, desc="LIME-image", leave=False):
        img_hwc = np.transpose(img, (1, 2, 0))
        exp = explainer.explain_instance(
            img_hwc,
            predict_fn,
            top_labels=1,
            hide_color=0,
            num_samples=lime_num_samples,
        )
        label = int(np.argmax(predict_fn(img_hwc[None, ...])[0]))
        segments = exp.segments
        local_map = np.zeros(segments.shape, dtype=np.float64)
        for seg_id, weight in exp.local_exp[label]:
            local_map[segments == seg_id] = abs(weight)
        dense = np.repeat(local_map[None, ...], 3, axis=0)[None, ...]
        pf, _, _ = images_to_patch_features(dense, patch=patch_size, channel_reduce="keep")
        v = pf.reshape(-1)
        if scores is None:
            scores = np.zeros_like(v)
        scores += v

    return scores / max(len(images), 1)

# ---------------------------
# Faithfulness
# ---------------------------
def deletion_curve_for_patch_ranking(model, X_images, y_true, patch_scores, device, patch_size=4, steps=10):
    rank = rank_indices(patch_scores)
    _, _, h, w = X_images.shape
    patches_per_row = h // patch_size
    patches_per_col = w // patch_size

    def zero_patch_batch(Xb, patch_id):
        Xb = Xb.copy()
        ch = patch_id // (patches_per_row * patches_per_col)
        rem = patch_id % (patches_per_row * patches_per_col)
        i = rem // patches_per_col
        j = rem % patches_per_col
        Xb[:, ch, i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size] = 0.0
        return Xb

    fracs = np.linspace(0.0, 1.0, steps)
    accs = []

    with torch.no_grad():
        for frac in fracs:
            k = int(frac * len(rank))
            current = X_images.copy()
            for p in rank[:k]:
                current = zero_patch_batch(current, int(p))
            logits = []
            bs = 128
            for s in range(0, len(current), bs):
                xb = torch.from_numpy(current[s:s+bs]).float().to(device)
                logits.append(model(xb).cpu().numpy())
            preds = np.concatenate(logits).argmax(axis=1)
            accs.append(accuracy_score(y_true, preds))

    area = np.trapezoid(accs, fracs)
    return {
        "deletion_area": float(area),
        "acc_start": float(accs[0]),
        "acc_end": float(accs[-1]),
    }

def topk_sufficiency_text(clf, X_val_sparse, y_val, scores, topk=100):
    idx = rank_indices(scores)[:topk]

    X_masked = X_val_sparse.copy().tolil()
    all_idx = np.arange(X_val_sparse.shape[1])
    remove_idx = np.setdiff1d(all_idx, idx)

    X_masked[:, remove_idx] = 0
    X_masked = X_masked.tocsr()

    pred = clf.predict(X_masked)
    return float(accuracy_score(y_val, pred))

# ---------------------------
# Text: 20 Newsgroups
# ---------------------------
def load_text_dataset(max_docs=6000, seed=42):
    cats = [
        "sci.space",
        "comp.graphics",
        "rec.sport.baseball",
        "talk.politics.misc",
    ]
    train = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))
    test = fetch_20newsgroups(subset="test", categories=cats, remove=("headers", "footers", "quotes"))

    x_train = train.data[:max_docs]
    y_train = np.array(train.target[:max_docs])
    x_test = test.data[: max_docs // 2]
    y_test = np.array(test.target[: max_docs // 2])
    return x_train, y_train, x_test, y_test, train.target_names

def train_text_model(x_train, y_train, x_val, y_val, max_features=10000):
    vec = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=3,
    )
    Xtr = vec.fit_transform(x_train)
    Xva = vec.transform(x_val)

    clf = LogisticRegression(max_iter=2000, multi_class="auto", solver="lbfgs")
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xva)
    acc = accuracy_score(y_val, pred)
    return clf, vec, {"val_acc": float(acc)}

def text_token_dropout(texts, drop_prob=0.15, seed=42):
    rng = np.random.default_rng(seed)
    out = []
    for text in texts:
        toks = text.split()
        kept = [t for t in toks if rng.random() > drop_prob]
        out.append(" ".join(kept) if kept else text)
    return out

# ---------------------------
# Method comparison table
# ---------------------------
def compare_methods(score_map, topk=10):
    rows = []
    names = list(score_map.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            sa, sb = np.asarray(score_map[a]), np.asarray(score_map[b])
            rho = np.nan_to_num(stats.spearmanr(sa, sb).statistic)
            tau = np.nan_to_num(stats.kendalltau(rank_indices(sa), rank_indices(sb)).statistic)
            ov = topk_overlap(rank_indices(sa), rank_indices(sb), k=topk)
            rows.append({
                "method_a": a,
                "method_b": b,
                "spearman": float(rho),
                "kendall": float(tau),
                f"top{topk}_overlap": float(ov)
            })
    return pd.DataFrame(rows)

# ---------------------------
# Vision runner
# ---------------------------
def run_vision(cfg):
    device = torch.device(cfg["device"])
    train_loader, val_loader, test_loader = make_cifar_dataloaders(
        cfg["data_dir"],
        batch_size=cfg["batch_size"],
        train_subset=cfg["train_subset"],
        val_subset=cfg["val_subset"],
        test_subset=cfg["test_subset"],
        seed=cfg["seed"],
    )

    model = make_resnet18(num_classes=10)
    model, train_info = train_torch_classifier(
        model, train_loader, val_loader, device, epochs=cfg["epochs"], lr=cfg["lr"]
    )

    val_acc = eval_torch_accuracy(model, val_loader, device)
    test_acc = eval_torch_accuracy(model, test_loader, device)

    X_val_img, y_val, logits_val = collect_logits_and_inputs(model, val_loader, device)
    X_val_feat, feature_names, groups = images_to_patch_features(
        X_val_img, patch=cfg["patch_size"], channel_reduce="keep"
    )

    excir, excir_time = timed(excir_scores, X_val_feat, logits_val)
    mi, mi_time = timed(mutual_info_classif, X_val_feat, y_val, random_state=cfg["seed"])
    cc_scores = {f"class_{c}": cc_cir_scores(X_val_feat, logits_val, c) for c in range(logits_val.shape[1])}
    block_scores, block_time = timed(blockcir_scores, X_val_feat, logits_val, groups)

    score_map = {
        "ExCIR": excir,
        "MI": np.asarray(mi),
    }
    runtimes = {
        "ExCIR": excir_time,
        "BlockCIR": block_time,
        "MI": mi_time,
    }

    X_val_tensor = torch.from_numpy(X_val_img).float()

    if cfg["run_shap"]:
        if HAS_SHAP:
            shap_scores, shap_time = timed(
                shap_image_patch_scores,
                model,
                X_val_tensor[: min(16, len(X_val_tensor))],
                X_val_tensor,
                device,
                cfg["patch_size"],
                cfg["shap_samples"],
            )
            score_map["SHAP"] = shap_scores
            runtimes["SHAP"] = shap_time
        else:
            warnings.warn("SHAP not installed; skipping.")

    if cfg["run_lime"]:
        if HAS_LIME:
            lime_scores, lime_time = timed(
                lime_image_patch_scores,
                model,
                X_val_img,
                device,
                cfg["patch_size"],
                cfg["lime_samples"],
                cfg["lime_num_samples"],
            )
            score_map["LIME"] = lime_scores
            runtimes["LIME"] = lime_time
        else:
            warnings.warn("LIME not installed; skipping.")

    stability = {
        "ExCIR": bootstrap_ranking_stability(
            X_val_feat, logits_val, lambda X, Y: excir_scores(X, Y),
            n_boot=cfg["n_boot"], topk=cfg["topk"], seed=cfg["seed"]
        ),
        "MI": bootstrap_ranking_stability(
            X_val_feat, y_val[:, None], lambda X, Y: mutual_info_classif(X, Y[:, 0], random_state=cfg["seed"]),
            n_boot=cfg["n_boot"], topk=cfg["topk"], seed=cfg["seed"]
        ),
    }

    X_noisy = add_gaussian_noise_images(X_val_img, sigma=cfg["image_noise_sigma"])
    with torch.no_grad():
        noisy_logits = []
        for s in range(0, len(X_noisy), cfg["batch_size"]):
            xb = torch.from_numpy(X_noisy[s:s+cfg["batch_size"]]).float().to(device)
            noisy_logits.append(model(xb).cpu().numpy())
    noisy_logits = np.concatenate(noisy_logits)
    X_noisy_feat, _, _ = images_to_patch_features(X_noisy, patch=cfg["patch_size"], channel_reduce="keep")
    excir_noisy = excir_scores(X_noisy_feat, noisy_logits)

    noise_stats = {
        "ExCIR_spearman_clean_vs_noisy": float(np.nan_to_num(stats.spearmanr(excir, excir_noisy).statistic)),
        f"ExCIR_top{cfg['topk']}_overlap_clean_vs_noisy": float(topk_overlap(rank_indices(excir), rank_indices(excir_noisy), k=cfg["topk"])),
    }

    mi_agreement = {
        "ExCIR_vs_MI_spearman": float(np.nan_to_num(stats.spearmanr(excir, mi).statistic)),
        "ExCIR_vs_MI_kendall": float(np.nan_to_num(stats.kendalltau(rank_indices(excir), rank_indices(mi)).statistic)),
    }

    deletion = deletion_curve_for_patch_ranking(
        model, X_val_img, y_val, excir, device, patch_size=cfg["patch_size"], steps=cfg["deletion_steps"]
    )

    lw_results = []
    if cfg["run_lw"]:
        criteria = LWCriteria(
            proj_threshold=cfg["proj_threshold"],
            mmd_threshold=cfg["mmd_threshold"],
            kl_threshold=cfg["kl_threshold"],
            risk_gap_threshold=cfg["risk_gap_threshold"],
            spearman_threshold=cfg["lw_spearman_threshold"],
            topk_threshold=cfg["lw_topk_threshold"],
        )

        full_scores = excir
        full_metric = val_acc

        for frac in [0.20, 0.30, 0.35, 0.40, 0.50]:
            subset_size = max(1000, int(frac * len(train_loader.dataset)))
            tr_frac, va_frac, _ = make_cifar_dataloaders(
                cfg["data_dir"],
                batch_size=cfg["batch_size"],
                train_subset=subset_size,
                val_subset=cfg["val_subset"],
                test_subset=cfg["test_subset"],
                seed=cfg["seed"],
            )
            lw_model = make_resnet18(num_classes=10)
            lw_model, _ = train_torch_classifier(
                lw_model, tr_frac, va_frac, device, epochs=cfg["epochs_lw"], lr=cfg["lr"]
            )
            lw_val_acc = eval_torch_accuracy(lw_model, val_loader, device)
            X_lw_img, _, lw_logits = collect_logits_and_inputs(lw_model, val_loader, device)
            X_lw_feat, _, _ = images_to_patch_features(X_lw_img, patch=cfg["patch_size"], channel_reduce="keep")
            lw_scores = excir_scores(X_lw_feat, lw_logits)
            gate = evaluate_lw_gate(
                X_val_feat, X_lw_feat, full_scores, lw_scores, full_metric, lw_val_acc, criteria, topk=cfg["topk"]
            )
            lw_results.append({
                "fraction": frac,
                "subset_size": subset_size,
                "val_acc": lw_val_acc,
                **gate
            })

    pairwise = compare_methods(score_map, topk=cfg["topk"])

    results = {
        "track": "vision",
        "val_acc": val_acc,
        "test_acc": test_acc,
        "train_info": train_info,
        "feature_ranking_ExCIR": rank_df(excir, feature_names).head(30).to_dict(orient="records"),
        "block_scores": block_scores,
        "cc_top_features": {
            c: rank_df(sc, feature_names).head(10).to_dict(orient="records")
            for c, sc in cc_scores.items()
        },
        "pairwise_method_comparison": pairwise.to_dict(orient="records"),
        "runtimes_seconds": runtimes,
        "stability": stability,
        "noise_robustness": noise_stats,
        "mi_agreement": mi_agreement,
        "deletion_curve": deletion,
        "lw_results": lw_results,
    }
    return results

# ---------------------------
# Text runner
# ---------------------------
def run_text(cfg):
    x_train, y_train, x_test, y_test, class_names = load_text_dataset(
        max_docs=cfg["text_max_docs"], seed=cfg["seed"]
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_train, y_train, test_size=0.2, random_state=cfg["seed"], stratify=y_train
    )

    clf, vec, info = train_text_model(
        x_train, y_train, x_val, y_val, max_features=cfg["max_text_features"]
    )

    X_val_sparse = vec.transform(x_val)
    X_test_sparse = vec.transform(x_test)
    val_probs = clf.predict_proba(X_val_sparse)
    test_probs = clf.predict_proba(X_test_sparse)
    val_acc = accuracy_score(y_val, clf.predict(X_val_sparse))
    test_acc = accuracy_score(y_test, clf.predict(X_test_sparse))

    feature_names = vec.get_feature_names_out()
    X_val_dense = X_val_sparse.toarray()

    excir, excir_time = timed(excir_scores, X_val_dense, val_probs)
    mi, mi_time = timed(mutual_info_classif, X_val_dense, y_val, random_state=cfg["seed"])
    cc_scores = {f"class_{c}": cc_cir_scores(X_val_dense, val_probs, c) for c in range(val_probs.shape[1])}

    score_map = {
        "ExCIR": excir,
        "MI": np.asarray(mi),
    }
    runtimes = {
        "ExCIR": excir_time,
        "MI": mi_time,
    }

    if cfg["run_shap"]:
        if HAS_SHAP:
            shap_scores, shap_time = timed(
                shap_text_scores_linear,
                clf,
                X_val_sparse,
                cfg["shap_samples"],
            )
            score_map["SHAP"] = shap_scores
            runtimes["SHAP"] = shap_time
        else:
            warnings.warn("SHAP not installed; skipping.")

    if cfg["run_lime"]:
        if HAS_LIME:
            lime_scores, lime_time = timed(
                lime_text_global_scores,
                clf,
                vec,
                x_val,
                class_names,
                cfg["lime_samples"],
                cfg["text_lime_features"],
                cfg["seed"],
            )
            score_map["LIME"] = lime_scores
            runtimes["LIME"] = lime_time
        else:
            warnings.warn("LIME not installed; skipping.")

    stability = {
        "ExCIR": bootstrap_ranking_stability(
            X_val_dense, val_probs, lambda X, Y: excir_scores(X, Y),
            n_boot=cfg["n_boot"], topk=cfg["topk"], seed=cfg["seed"]
        ),
        "MI": bootstrap_ranking_stability(
            X_val_dense, y_val[:, None], lambda X, Y: mutual_info_classif(X, Y[:, 0], random_state=cfg["seed"]),
            n_boot=cfg["n_boot"], topk=cfg["topk"], seed=cfg["seed"]
        ),
    }

    x_val_noisy = text_token_dropout(x_val, drop_prob=cfg["text_drop_prob"], seed=cfg["seed"])
    X_val_noisy_sparse = vec.transform(x_val_noisy)
    val_probs_noisy = clf.predict_proba(X_val_noisy_sparse)
    X_val_noisy_dense = X_val_noisy_sparse.toarray()
    excir_noisy = excir_scores(X_val_noisy_dense, val_probs_noisy)

    noise_stats = {
        "ExCIR_spearman_clean_vs_token_dropout": float(np.nan_to_num(stats.spearmanr(excir, excir_noisy).statistic)),
        f"ExCIR_top{cfg['topk']}_overlap_clean_vs_token_dropout": float(topk_overlap(rank_indices(excir), rank_indices(excir_noisy), k=cfg["topk"])),
    }

    mi_agreement = {
        "ExCIR_vs_MI_spearman": float(np.nan_to_num(stats.spearmanr(excir, mi).statistic)),
        "ExCIR_vs_MI_kendall": float(np.nan_to_num(stats.kendalltau(rank_indices(excir), rank_indices(mi)).statistic)),
    }

    lw_results = []
    if cfg["run_lw"]:
        criteria = LWCriteria(
            proj_threshold=cfg["proj_threshold"],
            mmd_threshold=cfg["mmd_threshold"],
            kl_threshold=cfg["kl_threshold"],
            risk_gap_threshold=cfg["risk_gap_threshold"],
            spearman_threshold=cfg["lw_spearman_threshold"],
            topk_threshold=cfg["lw_topk_threshold"],
        )

        full_scores = excir
        full_metric = val_acc
        full_vocab = vec.get_feature_names_out()

        for frac in [0.20, 0.30, 0.35, 0.40, 0.50]:
            keep = max(500, int(frac * len(x_train)))
            x_sub = x_train[:keep]
            y_sub = y_train[:keep]

            lw_clf, lw_vec, _ = train_text_model(
                x_sub, y_sub, x_val, y_val, max_features=cfg["max_text_features"]
            )
            X_lw_val_sparse = lw_vec.transform(x_val)
            lw_probs = lw_clf.predict_proba(X_lw_val_sparse)
            lw_acc = accuracy_score(y_val, lw_clf.predict(X_lw_val_sparse))
            X_lw_dense = X_lw_val_sparse.toarray()
            lw_scores = excir_scores(X_lw_dense, lw_probs)

            lw_vocab = lw_vec.get_feature_names_out()
            shared = sorted(set(full_vocab) & set(lw_vocab))
            if len(shared) < 100:
                continue

            full_map = {w: i for i, w in enumerate(full_vocab)}
            lw_map = {w: i for i, w in enumerate(lw_vocab)}
            full_idx = np.array([full_map[w] for w in shared])
            lw_idx = np.array([lw_map[w] for w in shared])

            gate = evaluate_lw_gate(
                X_val_dense[:, full_idx],
                X_lw_dense[:, lw_idx],
                full_scores[full_idx],
                lw_scores[lw_idx],
                full_metric,
                lw_acc,
                criteria,
                topk=min(cfg["topk"], len(shared)),
            )
            lw_results.append({
                "fraction": frac,
                "subset_size": keep,
                "val_acc": lw_acc,
                **gate
            })

    pairwise = compare_methods(score_map, topk=cfg["topk"])
    suff = topk_sufficiency_text(clf, X_val_sparse, y_val, excir, topk=100)

    results = {
        "track": "text",
        "val_acc": float(val_acc),
        "test_acc": float(test_acc),
        "train_info": info,
        "feature_ranking_ExCIR": rank_df(excir, feature_names).head(50).to_dict(orient="records"),
        "cc_top_features": {
            c: rank_df(sc, feature_names).head(10).to_dict(orient="records")
            for c, sc in cc_scores.items()
        },
        "pairwise_method_comparison": pairwise.to_dict(orient="records"),
        "runtimes_seconds": runtimes,
        "stability": stability,
        "noise_robustness": noise_stats,
        "mi_agreement": mi_agreement,
        "lw_results": lw_results,
        "sufficiency_topk_100": suff,
    }
    return results

# ---------------------------
# Main runner
# ---------------------------
def run_benchmark(cfg=None):
    if cfg is None:
        cfg = CONFIG
    set_seed(cfg["seed"])
    Path(cfg["out_dir"]).mkdir(parents=True, exist_ok=True)

    if cfg["track"] == "vision":
        results = run_vision(cfg)
    elif cfg["track"] == "text":
        results = run_text(cfg)
    else:
        raise ValueError("CONFIG['track'] must be 'vision' or 'text'")

    out_file = Path(cfg["out_dir"]) / f"{cfg['track']}_results.json"
    with open(out_file, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Saved results to: {out_file}")
    print(json.dumps({
        "track": results["track"],
        "val_acc": results["val_acc"],
        "test_acc": results["test_acc"],
        "mi_agreement": results["mi_agreement"],
        "runtimes_seconds": results["runtimes_seconds"],
    }, indent=2))
    return results

# ---------------------------
# Run here
# ---------------------------
results = run_benchmark(CONFIG)

Saved results to: results_revision/vision_results.json
{
  "track": "vision",
  "val_acc": 0.6912,
  "test_acc": 0.6952,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.020716911266039122,
    "ExCIR_vs_MI_kendall": -0.23156631762652707
  },
  "runtimes_seconds": {
    "ExCIR": 0.16118927899970004,
    "BlockCIR": 0.26694749199941725,
    "MI": 3.2359003999999914
  }
}


In [ ]:
CONFIG["track"] = "vision"
CONFIG["epochs"] = 15
CONFIG["epochs_lw"] = 5
CONFIG["train_subset"] = 20000
CONFIG["val_subset"] = 5000
CONFIG["test_subset"] = 5000
CONFIG["batch_size"] = 128
CONFIG["run_shap"] = True
CONFIG["run_lime"] = True
results = run_benchmark(CONFIG)

LIME-image:   0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

LIME-image:   6%|▋         | 1/16 [00:00<00:02,  6.11it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

LIME-image:  94%|█████████▍| 15/16 [00:01<00:00, 12.75it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

Saved results to: results_revision/vision_results.json
{
  "track": "vision",
  "val_acc": 0.5972,
  "test_acc": 0.6002,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.020803380083010067,
    "ExCIR_vs_MI_kendall": -0.2336387434554974
  },
  "runtimes_seconds": {
    "ExCIR": 0.15528262600037124,
    "BlockCIR": 0.2655736819997401,
    "MI": 3.144914661999792,
    "SHAP": 30.11939951199929,
    "LIME": 1.3356230050003433
  }
}


In [ ]:
CONFIG["track"] = "text"
CONFIG["run_shap"] = True
CONFIG["run_lime"] = True
results = run_benchmark(CONFIG)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

Saved results to: results_revision/text_results.json
{
  "track": "text",
  "val_acc": 0.8928571428571429,
  "test_acc": 0.8536912751677852,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.04798365109069605,
    "ExCIR_vs_MI_kendall": 0.009899069906990698
  },
  "runtimes_seconds": {
    "ExCIR": 0.5871192430001884,
    "MI": 25.51963937200003,
    "SHAP": 0.03264133700008642,
    "LIME": 3.8400535890000356
  }
}


In [ ]:
CONFIG["track"] = "vision"
CONFIG["epochs"] = 15
CONFIG["epochs_lw"] = 5
CONFIG["train_subset"] = 20000
CONFIG["val_subset"] = 5000
CONFIG["test_subset"] = 5000
CONFIG["run_lw"] = True
CONFIG["run_shap"] = False
CONFIG["run_lime"] = False

CONFIG["track"] = "vision"
vision_results = run_benchmark(CONFIG)

CONFIG["track"] = "text"
text_results = run_benchmark(CONFIG)

all_results = {
    "vision": vision_results,
    "text": text_results
}

print(json.dumps(all_results["vision"]["lw_results"], indent=2))

Saved results to: results_revision/vision_results.json
{
  "track": "vision",
  "val_acc": 0.5972,
  "test_acc": 0.6002,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.020803380083010067,
    "ExCIR_vs_MI_kendall": -0.2336387434554974
  },
  "runtimes_seconds": {
    "ExCIR": 0.15886116399997263,
    "BlockCIR": 0.266233387000284,
    "MI": 3.14355710000018
  }
}


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

Saved results to: results_revision/text_results.json
{
  "track": "text",
  "val_acc": 0.8928571428571429,
  "test_acc": 0.8536912751677852,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.04798365109069605,
    "ExCIR_vs_MI_kendall": 0.009899069906990698
  },
  "runtimes_seconds": {
    "ExCIR": 0.6039952970004379,
    "MI": 25.986825683999996
  }
}
[
  {
    "fraction": 0.2,
    "subset_size": 4000,
    "val_acc": 0.4154,
    "proj": 0.0,
    "mmd": 0.0,
    "kl": 0.0,
    "spearman": 0.9995557876461495,
    "topk_overlap": 0.8181818181818182,
    "risk_gap": 0.18179999999999996,
    "passed": false
  },
  {
    "fraction": 0.3,
    "subset_size": 6000,
    "val_acc": 0.4332,
    "proj": 0.0,
    "mmd": 0.0,
    "kl": 0.0,
    "spearman": 0.9970837967609799,
    "topk_overlap": 0.8181818181818182,
    "risk_gap": 0.16399999999999998,
    "passed": false
  },
  {
    "fraction": 0.35,
    "subset_size": 7000,
    "val_acc": 0.443,
    "proj": 0.0,
    "mmd": 0.0,
    "kl": 0.0,
   

In [ ]:
CONFIG["train_subset"] = None   # full train split
results = run_benchmark(CONFIG)
print(json.dumps(results["vision"]["lw_sweep"], indent=2))

In [ ]:
CONFIG["track"] = "vision"
CONFIG["run_lw"] = False
CONFIG["run_shap"] = False
CONFIG["run_lime"] = False

results = run_benchmark(CONFIG)

print(json.dumps(results["noise_robustness"], indent=2))

Saved results to: results_revision/vision_results.json
{
  "track": "vision",
  "val_acc": 0.5972,
  "test_acc": 0.6002,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.020803380083010067,
    "ExCIR_vs_MI_kendall": -0.2336387434554974
  },
  "runtimes_seconds": {
    "ExCIR": 0.15023645899964322,
    "BlockCIR": 0.23307483799908368,
    "MI": 2.8061500069998147
  }
}
{
  "ExCIR_spearman_clean_vs_noisy": 0.999577828717142,
  "ExCIR_top10_overlap_clean_vs_noisy": 0.8181818181818182
}


In [ ]:
CONFIG["track"] = "text"
CONFIG["run_lw"] = True
CONFIG["run_shap"] = True
CONFIG["run_lime"] = True

results = run_benchmark(CONFIG)

print(json.dumps(results["noise_robustness"], indent=2))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

Saved results to: results_revision/text_results.json
{
  "track": "text",
  "val_acc": 0.8928571428571429,
  "test_acc": 0.8536912751677852,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.04798365109069605,
    "ExCIR_vs_MI_kendall": 0.009899069906990698
  },
  "runtimes_seconds": {
    "ExCIR": 0.542444749000424,
    "MI": 23.12728881899966,
    "SHAP": 0.015315409998947871,
    "LIME": 0.7726832669995929
  }
}
{
  "ExCIR_spearman_clean_vs_token_dropout": 0.9575071000299157,
  "ExCIR_top10_overlap_clean_vs_token_dropout": 0.0
}


In [ ]:
CONFIG["track"] = "vision"
CONFIG["run_lw"] = False
CONFIG["run_shap"] = True
CONFIG["run_lime"] = True
CONFIG["shap_samples"] = 32
CONFIG["lime_samples"] = 6
CONFIG["lime_num_samples"] = 150

results = run_benchmark(CONFIG)
print(json.dumps(results["runtimes_seconds"], indent=2))
print(json.dumps(results["pairwise_method_comparison"][:5], indent=2))

LIME-image:   0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

LIME-image:  33%|███▎      | 2/6 [00:00<00:00, 19.48it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

LIME-image:  83%|████████▎ | 5/6 [00:00<00:00, 20.24it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

Saved results to: results_revision/vision_results.json
{
  "track": "vision",
  "val_acc": 0.5972,
  "test_acc": 0.6002,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.020803380083010067,
    "ExCIR_vs_MI_kendall": -0.2336387434554974
  },
  "runtimes_seconds": {
    "ExCIR": 0.15341923699998006,
    "BlockCIR": 0.28065004000018234,
    "MI": 3.1646816080010467,
    "SHAP": 15.148776485999406,
    "LIME": 0.29940055099905294
  }
}
{
  "ExCIR": 0.15341923699998006,
  "BlockCIR": 0.28065004000018234,
  "MI": 3.1646816080010467,
  "SHAP": 15.148776485999406,
  "LIME": 0.29940055099905294
}
[
  {
    "method_a": "ExCIR",
    "method_b": "MI",
    "spearman": -0.020803380083010067,
    "kendall": -0.2336387434554974,
    "top10_overlap": 0.0
  },
  {
    "method_a": "ExCIR",
    "method_b": "SHAP",
    "spearman": -0.06297133982584163,
    "kendall": 0.031086387434554975,
    "top10_overlap": 0.0
  },
  {
    "method_a": "ExCIR",
    "method_b": "LIME",
    "spearman": -0.34494601144754

In [ ]:
CONFIG["shap_samples"] = 16
CONFIG["lime_samples"] = 4
CONFIG["lime_num_samples"] = 100

In [ ]:
# --- VISION ---
CONFIG["track"] = "vision"
results_vision = run_benchmark(CONFIG)

# --- TEXT ---
CONFIG["track"] = "text"
results_text = run_benchmark(CONFIG)

# --- PRINT BOTH ---
print(json.dumps({
    "vision": {
        "val_acc": results_vision["val_acc"],
        "test_acc": results_vision["test_acc"],
        "mi_corr_spearman": results_vision["mi_agreement"],
        "noise": results_vision.get("noise_robustness", {}),
        "lw": results_vision.get("lw_results", []),
        "runtime": results_vision.get("runtimes_seconds", {})
    },
    "text": {
        "val_acc": results_text["val_acc"],
        "test_acc": results_text["test_acc"],
        "mi_corr_spearman": results_text["mi_agreement"],
        "noise": results_text.get("noise_robustness", {}),
        "lw": results_text.get("lw_results", []),
        "runtime": results_text.get("runtimes_seconds", {})
    }
}, indent=2))

LIME-image:   0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

LIME-image:  50%|█████     | 3/6 [00:00<00:00, 20.83it/s]

...

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

Saved results to: results_revision/vision_results.json
{
  "track": "vision",
  "val_acc": 0.5972,
  "test_acc": 0.6002,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.020803380083010067,
    "ExCIR_vs_MI_kendall": -0.2336387434554974
  },
  "runtimes_seconds": {
    "ExCIR": 0.15094679400135647,
    "BlockCIR": 0.2541178290011885,
    "MI": 3.2364450889999716,
    "SHAP": 14.965098079999734,
    "LIME": 0.2931537459990068
  }
}


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Saved results to: results_revision/text_results.json
{
  "track": "text",
  "val_acc": 0.8928571428571429,
  "test_acc": 0.8536912751677852,
  "mi_agreement": {
    "ExCIR_vs_MI_spearman": -0.04798365109069605,
    "ExCIR_vs_MI_kendall": 0.009899069906990698
  },
  "runtimes_seconds": {
    "ExCIR": 0.5870119190003606,
    "MI": 25.508251904000645,
    "SHAP": 0.01738304599894036,
    "LIME": 0.8630723179994675
  }
}
{
  "vision": {
    "val_acc": 0.5972,
    "test_acc": 0.6002,
    "mi_corr_spearman": {
      "ExCIR_vs_MI_spearman": -0.020803380083010067,
      "ExCIR_vs_MI_kendall": -0.2336387434554974
    },
    "noise": {
      "ExCIR_spearman_clean_vs_noisy": 0.9996473428641187,
      "ExCIR_top10_overlap_clean_vs_noisy": 1.0
    },
    "lw": [],
    "runtime": {
      "ExCIR": 0.15094679400135647,
      "BlockCIR": 0.2541178290011885,
      "MI": 3.2364450889999716,
      "SHAP": 14.965098079999734,
      "LIME": 0.2931537459990068
    }
  },
  "text": {
    "val_acc": 0.89285714